# Engenharia de Features com Regras Privadas

Notebook inicial para estruturar a engenharia de features sem expor listas sensíveis, thresholds ou regras de negócio diretamente no corpo do notebook. A definição completa das regras deve viver apenas em `config/engenharia_features.private.json`.

In [ ]:
import json
import sqlite3
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)


NOTEBOOK_DIR = Path.cwd()
PRIVATE_CONFIG_PATH = Path("../config/engenharia_features.private.json")


def load_json(path: Path) -> dict[str, Any]:
    """Lê um JSON com encoding explícito."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


CONFIG: dict[str, Any] = {}
CONFIG_READY = False
IO_CFG: dict[str, Any] = {}
CONSTANTS: dict[str, Any] = {}
DATASET_SPECS: list[dict[str, Any]] = []
SHARED_RULES: list[dict[str, Any]] = []
EXPORTS_CFG: dict[str, Any] = {}
AUDIT_CFG: dict[str, Any] = {}
INPUT_DB: Path | None = None
OUTPUT_DIR: Path | None = None
AUDIT_PATH: Path | None = None
VARIABLE_MAP_PATH: Path | None = None

if PRIVATE_CONFIG_PATH.exists():
    CONFIG = load_json(PRIVATE_CONFIG_PATH)
    IO_CFG = CONFIG.get("io", {}) or {}
    CONSTANTS = CONFIG.get("constants", {}) or {}
    DATASET_SPECS = CONFIG.get("datasets", []) or []
    SHARED_RULES = CONFIG.get("shared_rules", []) or []
    EXPORTS_CFG = CONFIG.get("exports", {}) or {}
    AUDIT_CFG = CONFIG.get("auditing", {}) or {}

    INPUT_DB = Path(IO_CFG.get("input_db", "")) if IO_CFG.get("input_db") else None
    OUTPUT_DIR = Path(IO_CFG.get("output_dir", "")) if IO_CFG.get("output_dir") else None
    AUDIT_PATH = Path(IO_CFG.get("audit_path", "")) if IO_CFG.get("audit_path") else None
    VARIABLE_MAP_PATH = Path(IO_CFG.get("variable_map_path", "")) if IO_CFG.get("variable_map_path") else None

    required_paths = {
        "input_db": INPUT_DB,
        "output_dir": OUTPUT_DIR,
        "audit_path": AUDIT_PATH,
        "variable_map_path": VARIABLE_MAP_PATH,
    }
    missing_paths = [name for name, value in required_paths.items() if value is None]
    missing_constants = [name for name in ("key_columns",) if not CONSTANTS.get(name)]

    if not missing_paths and not missing_constants:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        if AUDIT_PATH is not None:
            AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
        CONFIG_READY = True
    else:
        print("Configuração privada encontrada, mas ainda incompleta.")
        print(f"Campos pendentes: {missing_paths + missing_constants}")
else:
    print("Arquivo privado não encontrado. Crie config/engenharia_features.private.json para executar o pipeline.")

print(f"Config privado: {PRIVATE_CONFIG_PATH.resolve()}")
print(f"Config pronto: {CONFIG_READY}")
print(f"Datasets configurados: {len(DATASET_SPECS)}")
print(f"Regras compartilhadas: {len(SHARED_RULES)}")

In [ ]:
def load_dataset(conn: sqlite3.Connection, spec: dict[str, Any]) -> pd.DataFrame:
    """Carrega um dataset a partir da especificação do config."""
    query = spec.get("query") or f"SELECT * FROM {spec['table']}"
    df = pd.read_sql_query(query, conn)

    select_columns = [c for c in spec.get("select_columns", []) if c in df.columns]
    if select_columns:
        df = df.loc[:, select_columns]

    drop_columns = [c for c in spec.get("drop_columns", []) if c in df.columns]
    if drop_columns:
        df = df.drop(columns=drop_columns)

    return df.loc[:, ~df.columns.duplicated()].copy()


def apply_imputation(
    df: pd.DataFrame,
    imputation_spec: dict[str, list[str]] | None,
    constants: dict[str, Any],
) -> pd.DataFrame:
    """Aplica imputação declarativa sobre colunas numéricas e categóricas."""
    spec = imputation_spec or {}
    empty_value = constants.get("empty_category", "<EMPTY>")
    no_value = constants.get("no_category", "1")
    output = df.copy()

    numeric_zero = [c for c in spec.get("numeric_zero", []) if c in output.columns]
    numeric_median = [c for c in spec.get("numeric_median", []) if c in output.columns]
    categorical_empty = [c for c in spec.get("categorical_empty", []) if c in output.columns]
    categorical_no = [c for c in spec.get("categorical_no", []) if c in output.columns]

    for column in numeric_zero:
        output[column] = pd.to_numeric(output[column], errors="coerce").fillna(0)

    for column in numeric_median:
        numeric_series = pd.to_numeric(output[column], errors="coerce")
        output[column] = numeric_series.fillna(numeric_series.median())

    for column in categorical_empty:
        output[column] = output[column].astype("string").fillna(empty_value).astype("category")

    for column in categorical_no:
        output[column] = output[column].astype("string").fillna(no_value).astype("category")

    return output


def _normalized_op_value(series: pd.Series, value: Any) -> Any:
    if isinstance(value, str) and value in series.astype("string").dropna().unique():
        return value
    return value


def _compare_series(series: pd.Series, op: str, value: Any) -> pd.Series:
    if op in {"eq", "ne", "gt", "ge", "lt", "le"}:
        left = pd.to_numeric(series, errors="ignore")
        right = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
        if op == "eq":
            return left.eq(value) if isinstance(left.dtype, pd.StringDtype) else left.eq(right)
        if op == "ne":
            return left.ne(value) if isinstance(left.dtype, pd.StringDtype) else left.ne(right)
        if op == "gt":
            return pd.to_numeric(series, errors="coerce").gt(right)
        if op == "ge":
            return pd.to_numeric(series, errors="coerce").ge(right)
        if op == "lt":
            return pd.to_numeric(series, errors="coerce").lt(right)
        return pd.to_numeric(series, errors="coerce").le(right)

    if op == "in":
        values = list(value) if isinstance(value, (list, tuple, set)) else [value]
        return series.astype("string").isin([str(v) for v in values])

    if op == "not_in":
        values = list(value) if isinstance(value, (list, tuple, set)) else [value]
        return ~series.astype("string").isin([str(v) for v in values])

    raise ValueError(f"Operador não suportado: {op}")


def evaluate_condition_tree(df: pd.DataFrame, condition: dict[str, Any] | None) -> pd.Series:
    """Avalia uma árvore de condições declarativa."""
    if not condition:
        return pd.Series(True, index=df.index)

    if "clauses" in condition:
        logic = str(condition.get("logic", "all")).lower()
        clause_masks = [evaluate_condition_tree(df, clause) for clause in condition.get("clauses", [])]
        if not clause_masks:
            return pd.Series(True, index=df.index)
        if logic == "any":
            mask = clause_masks[0].copy()
            for clause_mask in clause_masks[1:]:
                mask = mask | clause_mask
            return mask
        mask = clause_masks[0].copy()
        for clause_mask in clause_masks[1:]:
            mask = mask & clause_mask
        return mask

    if "not" in condition:
        return ~evaluate_condition_tree(df, condition["not"])

    field = condition.get("field")
    if field not in df.columns:
        return pd.Series(False, index=df.index)

    op = str(condition.get("op", "eq")).lower()
    value = condition.get("value")
    return _compare_series(df[field], op, value)


def as_category_flag(mask: pd.Series) -> pd.Series:
    """Converte uma série booleana (ou já categórica 0/1) para flag categórica 0/1."""
    if isinstance(mask.dtype, pd.CategoricalDtype):
        mask = mask.astype("string")
    return mask.map({"1": True, "0": False, "True": True, "False": False}).fillna(False).astype("int8").astype("string").astype("category")


def apply_rule(
    df: pd.DataFrame,
    rule: dict[str, Any],
    constants: dict[str, Any],
    context_frames: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    """Aplica uma regra declarativa sobre um DataFrame."""
    output = df.copy()
    kind = rule.get("kind")
    outputs = rule.get("outputs", {})
    target_column = outputs.get("column")
    # Regras que transformam o DataFrame inteiro não precisam de coluna de saída.
    if not target_column and kind not in {"merge_context"}:
        raise ValueError(f"Regra sem coluna de saída: {rule.get('id', '<sem-id>')}")

    precondition = evaluate_condition_tree(output, rule.get("condition")) if target_column else pd.Series(True, index=output.index)

    if kind == "ratio":
        numerator = pd.to_numeric(output[rule["inputs"]["numerator"]], errors="coerce")
        denominator = pd.to_numeric(output[rule["inputs"]["denominator"]], errors="coerce")
        denominator = denominator.replace(0, pd.NA)
        ratio = numerator.div(denominator)
        output[target_column] = ratio.fillna(0).astype(outputs.get("dtype", "Float32"))

        threshold = rule.get("threshold")
        if threshold:
            threshold_mask = _compare_series(output[target_column], threshold.get("op", "gt"), threshold.get("value"))
            output[threshold.get("flag_column", f"{target_column}_flag")] = as_category_flag(precondition & threshold_mask)
        return output

    if kind == "difference":
        left = pd.to_numeric(output[rule["inputs"]["left"]], errors="coerce")
        right = pd.to_numeric(output[rule["inputs"]["right"]], errors="coerce")
        diff = left.sub(right)
        output[target_column] = diff.fillna(0).astype(outputs.get("dtype", "Float32"))
        return output

    if kind == "sum":
        columns = [col for col in rule.get("inputs", {}).get("columns", []) if col in output.columns]
        summed = sum(pd.to_numeric(output[col], errors="coerce").fillna(0) for col in columns)
        output[target_column] = summed.astype(outputs.get("dtype", "Float32"))
        return output

    if kind == "flag":
        output[target_column] = as_category_flag(precondition)
        return output

    if kind == "lookup_flag":
        keys = [col for col in rule.get("keys", []) if col in output.columns]
        matched_keys = set(map(tuple, output.loc[precondition, keys].drop_duplicates().to_numpy())) if keys else set()
        output[target_column] = as_category_flag(
            output[keys].apply(tuple, axis=1).isin(matched_keys) if keys else pd.Series(False, index=output.index)
        )
        return output

    if kind == "aggregate_merge":
        group_keys = [col for col in rule.get("group_keys", []) if col in output.columns]
        aggregations = rule.get("aggregations", [])
        if not group_keys or not aggregations:
            raise ValueError(f"Regra aggregate_merge inválida: {rule.get('id', '<sem-id>')}")

        agg_spec: dict[str, tuple[str, str]] = {}
        for agg in aggregations:
            source = agg["source"]
            alias = agg["alias"]
            func = agg.get("func", "sum")
            agg_spec[alias] = (source, func)

        aggregated = output.groupby(group_keys, as_index=False).agg(**agg_spec)
        return output.merge(aggregated, on=group_keys, how=rule.get("merge_how", "left"))

    if kind == "merge_context":
        source_name = rule.get("source_dataset")
        if source_name not in context_frames:
            raise ValueError(f"Dataset fonte não encontrado no contexto: {source_name}")
        source_df = context_frames[source_name]
        on_columns = [col for col in rule.get("on_columns", []) if col in output.columns and col in source_df.columns]
        if not on_columns:
            raise ValueError(f"Regra merge_context sem colunas de junção válidas: {rule.get('id', '<sem-id>')}")
        suffixes = rule.get("suffixes", (None, "_context"))
        merged = output.merge(source_df, on=on_columns, how=rule.get("how", "left"), suffixes=suffixes)
        # print(f"Merge context: {source_name} on {on_columns}, how={rule.get('how', 'left')}, suffixes={suffixes}")
        # display(merged)
        # display(output)
        
        # Remove colunas duplicadas provenientes do merge quando ambas têm o mesmo nome e sufixo vazio.
        if rule.get("drop_source_duplicates", True):
            duplicated_cols = [c for c in merged.columns if c.endswith(suffixes[1])]
            merged = merged.drop(columns=duplicated_cols)
        return merged

    fallback = rule.get("fallback_python")
    if fallback:
        local_env = {column: output[column] for column in output.columns}
        local_env.update({"output": output, "pd": pd, "np": np, "context_frames": context_frames})
        # Executa em escopo de função para que lambdas internas capturem pd/np/df.
        evaluated = eval(f"(lambda df, pd=pd, np=np, context_frames=context_frames: {fallback})(output, pd, np, context_frames)", {"__builtins__": __builtins__}, local_env)
        if isinstance(evaluated, pd.Series):
            output[target_column] = as_category_flag(evaluated)
        elif isinstance(evaluated, pd.DataFrame):
            output = evaluated
        else:
            output[target_column] = evaluated
        return output

    raise NotImplementedError(f"Kind não suportado: {kind}")


def sanitize_nulls(df: pd.DataFrame, empty_value: str = "<EMPTY>") -> pd.DataFrame:
    """Preenche valores nulos restantes para garantir saída consistente."""
    output = df.copy()
    numeric_cols = output.select_dtypes(include=["number"]).columns
    if len(numeric_cols) > 0:
        output[numeric_cols] = output[numeric_cols].fillna(0)

    cat_cols = output.select_dtypes(include=["category"]).columns
    for col in cat_cols:
        if empty_value not in output[col].cat.categories:
            output[col] = output[col].cat.add_categories([empty_value])
        output[col] = output[col].fillna(empty_value)

    text_cols = output.select_dtypes(include=["object", "string"]).columns
    text_cols = text_cols.difference(cat_cols)
    if len(text_cols) > 0:
        output[text_cols] = output[text_cols].astype("string").fillna(empty_value)
    return output


def apply_dataset_spec(
    df: pd.DataFrame,
    spec: dict[str, Any],
    constants: dict[str, Any],
    context_frames: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    """Aplica imputação e regras de um dataset."""
    output = apply_imputation(df, spec.get("imputation"), constants)

    for rule in spec.get("rules", []):
        if rule.get("enabled", True):
            output = apply_rule(output, rule, constants, context_frames)

    empty_value = constants.get("empty_category", "<EMPTY>")
    return sanitize_nulls(output, empty_value)


def build_audit_summary(dataset_frames: dict[str, pd.DataFrame]) -> dict[str, Any]:
    """Cria um resumo agregado da execução."""
    summary: dict[str, Any] = {
        "datasets": {},
        "total_rows": 0,
        "total_columns": 0,
    }

    for name, frame in dataset_frames.items():
        summary["datasets"][name] = {
            "rows": int(len(frame)),
            "columns": int(len(frame.columns)),
            "nulls": int(frame.isna().sum().sum()),
        }
        summary["total_rows"] += int(len(frame))
        summary["total_columns"] += int(len(frame.columns))

    return summary


def export_outputs(
    dataset_frames: dict[str, pd.DataFrame],
    exports_cfg: dict[str, Any],
    output_dir: Path,
    audit_path: Path,
    audit_summary: dict[str, Any],
) -> None:
    """Salva parquet e auditoria em disco."""
    output_dir.mkdir(parents=True, exist_ok=True)
    audit_path.parent.mkdir(parents=True, exist_ok=True)

    for export_spec in exports_cfg.get("parquet", []):
        dataset_name = export_spec.get("dataset")
        target_path = export_spec.get("path")
        if not dataset_name or dataset_name not in dataset_frames or not target_path:
            continue

        frame = dataset_frames[dataset_name]
        path = Path(target_path)
        if not path.is_absolute():
            path = output_dir / path
        path.parent.mkdir(parents=True, exist_ok=True)
        frame.to_parquet(path, index=bool(export_spec.get("index", False)))

    audit_path.write_text(json.dumps(audit_summary, ensure_ascii=False, indent=2), encoding="utf-8")


def execute_pipeline() -> dict[str, Any]:
    """Executa o fluxo principal quando a configuração estiver pronta."""
    if not CONFIG_READY:
        return {"frames": {}, "audit": {"status": "config_incompleta"}}

    dataset_frames: dict[str, pd.DataFrame] = {}
    with sqlite3.connect(str(INPUT_DB)) as conn:
        for spec in DATASET_SPECS:
            frame = load_dataset(conn, spec)
            dataset_frames[spec["name"]] = apply_dataset_spec(frame, spec, CONSTANTS, dataset_frames)

    audit_summary = build_audit_summary(dataset_frames)
    export_outputs(dataset_frames, EXPORTS_CFG, OUTPUT_DIR, AUDIT_PATH, audit_summary)

    return {"frames": dataset_frames, "audit": audit_summary}

## Registrar saídas e métricas

A última etapa consolida tempo de execução, volume processado e resumo de auditoria sem expor detalhes sensíveis das regras.

In [ ]:
from time import perf_counter


def run_notebook() -> dict[str, Any] | None:
    """Executa o pipeline principal quando a configuração estiver completa."""
    started = perf_counter()

    if not CONFIG_READY:
        print("Configuração incompleta. Preencha config/engenharia_features.private.json para executar o pipeline.")
        return None

    result = execute_pipeline()
    elapsed = perf_counter() - started
    print(f"Tempo total de execução: {elapsed:.2f}s")
    return result


pipeline_result = run_notebook()